In [3]:
# Tabellen für die TVD-Analyse erstellen
import pandas as pd
from pathlib import Path

from efficient_probit_regression.sampling import to_density
from efficient_probit_regression.total_variation_distance import total_variation_distance

dt = pd.read_pickle("Plots/probability_distributions.pkl")

rows = []

for _, row in dt.sort_values(["dataset", "p"]).iterrows():
    lev = row["prob_lp"]
    lw = row["prob_lewis_weight"]
    rnd = row["prob_random_evaluations_probabilities"]
    l2lp = row["prob_l2_lp_leverage_score"]

    rows.append({
        "dataset": row["dataset"],
        "p": float(row["p"]),
        "lp - lewis": round(total_variation_distance(lev, lw), 3),
        "lp - l2lp": round(total_variation_distance(lev, l2lp), 3),
        "lp - random": round(total_variation_distance(lev, rnd), 3),
        "l2lp - lewis": round(total_variation_distance(l2lp, lw), 3),
        "lewis - random": round(total_variation_distance(lw, rnd), 3),
        "l2lp - random": round(total_variation_distance(l2lp, rnd), 3),
    })

dt = pd.DataFrame(rows).sort_values(["p", "dataset"]).reset_index(drop=True)

out_dir = Path("Tabellen")
out_dir.mkdir(exist_ok=True, parents=True)
out_csv = "tvd_all_methods.csv"
dt.to_csv(out_dir / out_csv, index=False, float_format="%.10f")

print("Saved:", out_csv)
print("TVD Analyse:")
for p_value, df_p in dt.groupby("p", sort=False):
    print(f"\n=== p = {p_value} ===")
    print(df_p.reset_index(drop=True).to_string())

Saved: tvd_all_methods.csv
TVD Analyse:

=== p = 1.0 ===
     dataset    p  lp - lewis  lp - l2lp  lp - random  l2lp - lewis  lewis - random  l2lp - random
0  Covertype  1.0       0.189      0.173        0.260         0.068           0.376          0.349
1  Example2D  1.0       0.114      0.164        0.179         0.112           0.090          0.119
2       Iris  1.0       0.165      0.175        0.186         0.077           0.099          0.146
3     KDDCup  1.0       0.142      0.174        0.538         0.125           0.548          0.577
4    Webspam  1.0       0.090      0.139        0.248         0.108           0.267          0.324

=== p = 1.3 ===
     dataset    p  lp - lewis  lp - l2lp  lp - random  l2lp - lewis  lewis - random  l2lp - random
0  Covertype  1.3       0.236      0.238        0.251         0.051           0.385          0.390
1  Example2D  1.3       0.148      0.145        0.115         0.061           0.117          0.128
2       Iris  1.3       0.140      

In [1]:
# top_k Ergebnisse
import pandas as pd

print("Ergebnisse von top_k:")
top_k = pd.read_csv("Plots/BA_Comparison/TopK_thresholds_table.csv")
for dataset_name, p_value in top_k.sort_values(["dataset", "p"]).groupby("dataset", sort=False):
    print(f"\n=== Datensatz: {dataset_name} ===")
    for p_val, dataset_new_name in p_value.sort_values(["p", "dataset"]).groupby("p", sort=False):
        print(f"\n====== p = {p_val} ======")
        print(dataset_new_name.reset_index(drop=True).to_string())

Ergebnisse von top_k:

=== Datensatz: Covertype ===

====== p = 1.0 ======
     dataset    p                method     k50     k90     k95
0  Covertype  1.0    lp Leverage Scores  125407  452908  510845
1  Covertype  1.0         Lewis Weights   67162  423583  493312
2  Covertype  1.0    Random Evaluations  236549  501186  539481
3  Covertype  1.0  l2lp Leverage Scores   79910  417675  488600

====== p = 1.3 ======
     dataset    p                method     k50     k90     k95
0  Covertype  1.3    lp Leverage Scores  135837  455350  511606
1  Covertype  1.3         Lewis Weights   62828  410885  483989
2  Covertype  1.3    Random Evaluations  219538  493183  534724
3  Covertype  1.3  l2lp Leverage Scores   61306  397674  474602

====== p = 1.5 ======
     dataset    p                method     k50     k90     k95
0  Covertype  1.5    lp Leverage Scores  101776  430457  495942
1  Covertype  1.5         Lewis Weights   60915  403596  478488
2  Covertype  1.5    Random Evaluations  208159

In [3]:
# Bestimmung von c = max(np.maximum(lewis_prob / l2_lp_prob, l2_lp_prob / lewis_prob))
import pandas as pd

bestimmung_c = pd.read_csv("Tabellen/Bestimmung_c.csv")
print("Bestimmung von c = max(np.maximum(lewis_prob / l2_lp_prob, l2_lp_prob / lewis_prob)):")

for dataset_name, p_value in bestimmung_c.sort_values(["dataset", "p"]).groupby("dataset", sort=False):
    print(f"\n=== Datensatz: {dataset_name} ===")
    print(p_value.reset_index(drop=True).to_string())

Bestimmung von c = max(np.maximum(lewis_prob / l2_lp_prob, l2_lp_prob / lewis_prob)):

=== Datensatz: Covertype ===
     dataset    p         c
0  Covertype  1.0  105.1547
1  Covertype  1.3  175.4905
2  Covertype  1.5   49.5358
3  Covertype  1.7   38.7712
4  Covertype  2.0    1.0000

=== Datensatz: Example2D ===
     dataset    p       c
0  Example2D  1.0  1.7510
1  Example2D  1.3  1.4249
2  Example2D  1.5  1.3219
3  Example2D  1.7  1.1734
4  Example2D  2.0  1.0000

=== Datensatz: Iris ===
  dataset    p       c
0    Iris  1.0  1.8384
1    Iris  1.3  1.6014
2    Iris  1.5  1.3682
3    Iris  1.7  1.2318
4    Iris  2.0  1.0000

=== Datensatz: KDDCup ===
  dataset    p       c
0  KDDCup  1.0  4.3332
1  KDDCup  1.3  4.5074
2  KDDCup  1.5  3.0530
3  KDDCup  1.7  2.1152
4  KDDCup  2.0  1.0000

=== Datensatz: Webspam ===
   dataset    p       c
0  Webspam  1.0  7.7185
1  Webspam  1.3  6.5425
2  Webspam  1.5  4.0277
3  Webspam  1.7  2.3446
4  Webspam  2.0  1.0000


In [4]:
# Bestimmung von c = max(lewis_prob / l2_lp_prob)
import pandas as pd

bestimmung_c = pd.read_csv("Tabellen/Bestimmung_c_max.csv")
print("Bestimmung von c = max(lewis_prob / l2_lp_prob):")

for dataset_name, p_value in bestimmung_c.sort_values(["dataset", "p"]).groupby("dataset", sort=False):
    print(f"\n=== Datensatz: {dataset_name} ===")
    print(p_value.reset_index(drop=True).to_string())

Bestimmung von c = max(lewis_prob / l2_lp_prob):

=== Datensatz: Covertype ===
     dataset    p         c
0  Covertype  1.0   64.2269
1  Covertype  1.3   20.4723
2  Covertype  1.5  149.1790
3  Covertype  1.7   11.0694
4  Covertype  2.0    1.0000

=== Datensatz: Example2D ===
     dataset    p       c
0  Example2D  1.0  1.7727
1  Example2D  1.3  1.5023
2  Example2D  1.5  1.2968
3  Example2D  1.7  1.1827
4  Example2D  2.0  1.0000

=== Datensatz: Iris ===
  dataset    p       c
0    Iris  1.0  1.4451
1    Iris  1.3  1.6425
2    Iris  1.5  1.3634
3    Iris  1.7  1.2311
4    Iris  2.0  1.0000

=== Datensatz: KDDCup ===
  dataset    p       c
0  KDDCup  1.0  4.3164
1  KDDCup  1.3  4.6267
2  KDDCup  1.5  3.2226
3  KDDCup  1.7  2.1113
4  KDDCup  2.0  1.0000

=== Datensatz: Webspam ===
   dataset    p       c
0  Webspam  1.0  4.8725
1  Webspam  1.3  5.8627
2  Webspam  1.5  3.8766
3  Webspam  1.7  2.4776
4  Webspam  2.0  1.0000


In [5]:
# Vergleich von min(l_ip, l_i2) * 0.5 <= w_i <= max(l_ip, l_i2) * 2
import pandas as pd

vergleich_c_l2lp = pd.read_csv("Tabellen/Vergleich_minmax_l2lp_vs_lewis.csv")
print("Vergleich von min(l_ip, l_i2) * 0.5 <= w_i <= max(l_ip, l_i2) * 2:")

for p_value, dataset_name in vergleich_c_l2lp.sort_values(["p", "dataset"]).groupby("p", sort=False):
    print(f"\n=== p = {p_value} ===")
    print(dataset_name.reset_index(drop=True).to_string())

Vergleich von min(l_ip, l_i2) * 0.5 <= w_i <= max(l_ip, l_i2) * 2:

=== p = 1.0 ===
     dataset    p       n  true_scores  false_scores  ratio_true_scores  true_prob  false_prob  ratio_true_prob
0  Covertype  1.0  581012       581000            12             1.0000     580983          29           1.0000
1  Example2D  1.0     175          175             0             1.0000        175           0           1.0000
2       Iris  1.0     150          150             0             1.0000        150           0           1.0000
3     KDDCup  1.0  494021       466787         27234             0.9449     493983          38           0.9999
4    Webspam  1.0  350000       349673           327             0.9991     349692         308           0.9991

=== p = 1.3 ===
     dataset    p       n  true_scores  false_scores  ratio_true_scores  true_prob  false_prob  ratio_true_prob
0  Covertype  1.3  581012       581003             9             1.0000     581003           9           1.0000
1  

In [6]:
# Bestimme die maximale Abweichung von lewis zu max(l_ip, l_i2)
import pandas as pd

vergleich_c_lewis_l2lp = pd.read_csv("Tabellen/Maximale_Abweichung_lewis_max(l2,lp).csv")
print("Bestimme die maximale Abweichung von lewis zu max(l_ip, l_i2):")

for p_value, dataset_name in vergleich_c_lewis_l2lp.sort_values(["p", "dataset"]).groupby("p", sort=False):
    print(f"\n=== p = {p_value} ===")
    print(dataset_name.reset_index(drop=True).to_string())

Bestimme die maximale Abweichung von lewis zu max(l_ip, l_i2):

=== p = 1.0 ===
     dataset    p  max_scores_ratio  max_prob_ratio
0  Covertype  1.0           18.4689         18.6528
1  Example2D  1.0            1.6632          1.5346
2       Iris  1.0            1.9067          1.6193
3     KDDCup  1.0            4.6838          2.9788
4    Webspam  1.0            5.5236          5.5236

=== p = 1.3 ===
     dataset    p  max_scores_ratio  max_prob_ratio
0  Covertype  1.3            6.5177          6.6189
1  Example2D  1.3            1.4204          1.2640
2       Iris  1.3            1.5640          1.5640
3     KDDCup  1.3            5.7447          2.4188
4    Webspam  1.3            6.9736          5.4304

=== p = 1.5 ===
     dataset    p  max_scores_ratio  max_prob_ratio
0  Covertype  1.5          143.7859        144.4850
1  Example2D  1.5            1.2814          1.2814
2       Iris  1.5            1.3738          1.2796
3     KDDCup  1.5            3.4293          3.1107
4 